In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

## Ingestion del archivo language.csv

### Importar librerias

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType
from pyspark.sql.functions import lit, current_timestamp

### 1. Lectura del archivo

In [0]:
language_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .load(f"{bronze_folder_path}/language.csv")
)
display(language_df)

### Cargue de archivo + Definir schema 

In [0]:
language_schema = StructType([
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

language_df = (
    spark.read
        .schema(language_schema)
        .format("csv")
        .load(f"{bronze_folder_path}/language.csv")
)

### 2. Eliminar columnas 

In [0]:
language_df = language_df.drop("languageCode")
display(language_df)

### 3. Renombrar columnas

In [0]:
language_df  = language_df.withColumnsRenamed({"languageId": "language_id", "languageName": "language_name"})
display(language_df)

### 4. Crear columnas

In [0]:
language_df  = (
    add_ingestion_date(language_df) 
    .withColumns({"env": lit(v_env)})
)
display(language_df)

### 5. Escribir archivo

In [0]:
(language_df.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{silver_folder_path}/languages")
)

In [0]:
dbutils.notebook.exit("Success")